<a href="https://colab.research.google.com/github/mistyy12/flyrank-ML/blob/main/notebooks_finished/Copy_of_w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mistyy12/flyrank-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My project is a scoring and ranking task, where I use ML models to build ranked action queue. It assigns each eligible content page a score and order pages to help editors decide which to review first. The final queue includes suggested actions such as refresh, expand, protect, prune or monitor.

> Add blockquote





In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My starter target is is_declining_label: where its 1 if trend_direction is "down" and 0 otherwise. This rule-derived proxy based on current-window measurements. I will estimate the probability of this label and use it to review which pages should be reviewed first. It does not prove future decline or identify which action will improve a page.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("content_refresh_anonymized (1).csv")
df.head()
# is_declining_label = (trend_direction == "down")


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [34]:
# Check the page-level grain before removing duplicates
lane_df = df.loc[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

print("Duplicate content IDs:", lane_df["content_id"].duplicated().sum())

lane_df = lane_df.drop_duplicates(subset="content_id")


Duplicate content IDs: 0


In [8]:
df.columns.to_list()


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_df["is_declining_label"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

top20 = lane_df.head(20)
top20


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,ctr_90d,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0.007626,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0.000457,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0.000874,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0.004936,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,0.001254,1
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,8.5,0.00,25.00,0.0,good,page_1,down,-38.9,0.000252,1
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,7.0,0.00,0.00,0.0,low,page_1,down,-92.3,0.000000,1
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6,0.000580,0
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8,0.000890,1
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2,0.001613,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [29]:
precision20 = top20["is_declining_label"].mean()
print(f"Precision@20: {precision20:.2f}")

Precision@20: 0.65


I will use Precision@20 because it matches an assumed review capacity of 20 pages. It measures how many of the top 20 recommendations have positive target label. In here, 'good' or 'success' means outperforming a transparent baseline on the same held-out pages.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is a content page. Each row represents one unique content_id for a client and contains the features used to evaluate whether the page requires further action (such as refresh, prune, expand, etc)

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

columns = [
    "content_id",
    "client_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "is_declining_label"
]

print("Eligible pages:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())

display(lane_df[columns].head()) #Display selected columns)

Eligible pages: 30000
Unique content IDs: 30000


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
0,content_304f48230142,client_f369cb89fc,187,20,3803,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,463,22,11751,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,44.0,0.13,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

ML beats a fixed rule because the importance of one signal depends on other signals. For example, an old page with high exposure and weak engagement may need to be refreshed, while another old page that still performs well may not. These interactions make it difficult to define one fixed set of if-statements that works well across all pages. An ML model can learn these patterns from multiple features and rank pages more flexibly.




In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.